# 02 — Temporal Customer Inactivity Dataset

## FinPulse AI — Temporal Target Engineering

This notebook constructs a supervised temporal dataset to predict future transactional inactivity.

The objective is to estimate whether a previously active customer will perform no transactions during a future observation window.

The problem remains a binary classification problem:

- `0`: the customer performs at least one transaction during the future window;
- `1`: the customer performs no transactions during the future window.

Features will be calculated using only information available before each cutoff date.

The customer, account and loan lifecycle dates will not be used because the temporal quality audit identified systematic chronological inconsistencies in the synthetic source data.

The model will use the observed transaction history as its reliable temporal reference.

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

DB_USER = "finpulse_user"
DB_PASSWORD = "finpulse_password"
DB_HOST = "postgres"
DB_PORT = "5432"
DB_NAME = "finpulse"

engine = create_engine(
    f"postgresql+psycopg2://"
    f"{DB_USER}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

## 1. Transaction History Validation

Before defining cutoff dates and prediction windows, we validate the transaction period, number of customers and monthly activity distribution.

This step determines whether the available history can support a temporal inactivity target.

In [2]:
transaction_period_query = """
SELECT
    MIN(t.transaction_date) AS first_transaction_date,
    MAX(t.transaction_date) AS last_transaction_date,
    COUNT(*) AS total_transactions,
    COUNT(DISTINCT a.customer_id) AS customers_with_transactions
FROM dbt.stg_transactions t
INNER JOIN dbt.stg_accounts a
    ON t.account_id = a.account_id;
"""

transaction_period = pd.read_sql(
    transaction_period_query,
    engine
)

display(transaction_period)

,first_transaction_date,last_transaction_date,total_transactions,customers_with_transactions
0,2019-01-01 00:00:23,2025-12-31 23:55:27,1000000,38838


In [3]:
monthly_activity_query = """
SELECT
    DATE_TRUNC(
        'month',
        t.transaction_date
    )::date AS transaction_month,

    COUNT(*) AS total_transactions,

    COUNT(
        DISTINCT a.customer_id
    ) AS active_customers

FROM dbt.stg_transactions t

INNER JOIN dbt.stg_accounts a
    ON t.account_id = a.account_id

GROUP BY
    DATE_TRUNC(
        'month',
        t.transaction_date
    )::date

ORDER BY
    transaction_month;
"""

monthly_activity = pd.read_sql(
    monthly_activity_query,
    engine
)

print("Meses observados:", len(monthly_activity))

display(monthly_activity.head())
display(monthly_activity.tail())

Meses observados: 84


,transaction_month,total_transactions,active_customers
0,2019-01-01,12187,10059
1,2019-02-01,10944,9171
2,2019-03-01,12117,9979
3,2019-04-01,11616,9642
4,2019-05-01,12035,9910


,transaction_month,total_transactions,active_customers
79,2025-08-01,11938,9848
80,2025-09-01,11821,9772
81,2025-10-01,12158,10072
82,2025-11-01,11655,9681
83,2025-12-01,12031,9991


## 2. Temporal Target Window Comparison

To define a measurable inactivity target, we create quarterly cutoff dates.

For each customer and cutoff date:

1. The customer must have at least one transaction during the previous 180 days.
2. Only transactions before or on the cutoff date are used as historical information.
3. Future activity is observed during windows of 60, 90 and 180 days.
4. A customer is labeled inactive when no transaction occurs during the corresponding future window.

This comparison helps select a target window that is meaningful and statistically usable.

In [4]:
target_window_query = """
WITH cutoffs AS (

    SELECT
        (
            month_start +
            INTERVAL '1 month' -
            INTERVAL '1 day'
        )::date AS cutoff_date

    FROM GENERATE_SERIES(
        '2020-03-01'::date,
        '2025-06-01'::date,
        INTERVAL '3 months'
    ) AS month_start

),

customer_transactions AS (

    SELECT
        a.customer_id,
        t.transaction_date
    FROM dbt.stg_transactions t
    INNER JOIN dbt.stg_accounts a
        ON t.account_id = a.account_id

),

eligible_customers AS (

    SELECT
        c.cutoff_date,
        ct.customer_id,
        COUNT(*) AS transactions_previous_180d

    FROM cutoffs c

    INNER JOIN customer_transactions ct
        ON ct.transaction_date >=
            c.cutoff_date - INTERVAL '179 days'
       AND ct.transaction_date <
            c.cutoff_date + INTERVAL '1 day'

    GROUP BY
        c.cutoff_date,
        ct.customer_id

),

future_activity AS (

    SELECT
        e.cutoff_date,
        e.customer_id,
        e.transactions_previous_180d,

        COUNT(ct.transaction_date) FILTER (
            WHERE ct.transaction_date >=
                    e.cutoff_date + INTERVAL '1 day'
              AND ct.transaction_date <
                    e.cutoff_date + INTERVAL '61 days'
        ) AS transactions_next_60d,

        COUNT(ct.transaction_date) FILTER (
            WHERE ct.transaction_date >=
                    e.cutoff_date + INTERVAL '1 day'
              AND ct.transaction_date <
                    e.cutoff_date + INTERVAL '91 days'
        ) AS transactions_next_90d,

        COUNT(ct.transaction_date) FILTER (
            WHERE ct.transaction_date >=
                    e.cutoff_date + INTERVAL '1 day'
              AND ct.transaction_date <
                    e.cutoff_date + INTERVAL '181 days'
        ) AS transactions_next_180d

    FROM eligible_customers e

    LEFT JOIN customer_transactions ct
        ON ct.customer_id = e.customer_id
       AND ct.transaction_date >=
            e.cutoff_date + INTERVAL '1 day'
       AND ct.transaction_date <
            e.cutoff_date + INTERVAL '181 days'

    GROUP BY
        e.cutoff_date,
        e.customer_id,
        e.transactions_previous_180d

)

SELECT
    cutoff_date,
    customer_id,
    transactions_previous_180d,
    transactions_next_60d,
    transactions_next_90d,
    transactions_next_180d,

    CASE
        WHEN transactions_next_60d = 0 THEN 1
        ELSE 0
    END AS inactivity_60d_label,

    CASE
        WHEN transactions_next_90d = 0 THEN 1
        ELSE 0
    END AS inactivity_90d_label,

    CASE
        WHEN transactions_next_180d = 0 THEN 1
        ELSE 0
    END AS inactivity_180d_label

FROM future_activity

ORDER BY
    cutoff_date,
    customer_id;
"""

target_window_comparison = pd.read_sql(
    target_window_query,
    engine
)

print(
    "Shape:",
    target_window_comparison.shape
)

print(
    "Datas de corte:",
    target_window_comparison["cutoff_date"].nunique()
)

print(
    "Duplicidades cliente + cutoff:",
    target_window_comparison.duplicated(
        subset=["customer_id", "cutoff_date"]
    ).sum()
)

display(target_window_comparison.head())

Shape: (658587, 9)
Datas de corte: 22
Duplicidades cliente + cutoff: 0


,cutoff_date,customer_id,transactions_previous_180d,transactions_next_60d,transactions_next_90d,transactions_next_180d,inactivity_60d_label,inactivity_90d_label,inactivity_180d_label
0,2020-03-31,CUS000MKX5RHTAP,1,1,1,1,0,0,0
1,2020-03-31,CUS0015YUPYM2OQ,3,2,2,3,0,0,0
2,2020-03-31,CUS002V4AVJO5UQ,5,0,0,0,1,1,1
3,2020-03-31,CUS0031SK48C4X5,2,0,0,4,1,1,0
4,2020-03-31,CUS003E82EMAYB3,2,0,1,1,1,0,0


In [5]:
target_columns = [
    "inactivity_60d_label",
    "inactivity_90d_label",
    "inactivity_180d_label"
]

overall_target_distribution = pd.DataFrame({
    "inactive_customers": (
        target_window_comparison[target_columns]
        .sum()
    ),
    "total_customer_snapshots": len(
        target_window_comparison
    ),
    "inactivity_rate": (
        target_window_comparison[target_columns]
        .mean()
        .mul(100)
        .round(2)
    )
})

display(overall_target_distribution)

,inactive_customers,total_customer_snapshots,inactivity_rate
inactivity_60d_label,357966,658587,54.35
inactivity_90d_label,272462,658587,41.37
inactivity_180d_label,132315,658587,20.09


In [6]:
target_distribution_by_cutoff = (
    target_window_comparison
    .groupby("cutoff_date")[target_columns]
    .mean()
    .mul(100)
    .round(2)
)

target_distribution_by_cutoff.columns = [
    "inactivity_60d_percent",
    "inactivity_90d_percent",
    "inactivity_180d_percent"
]

display(target_distribution_by_cutoff)

,inactivity_60d_percent,inactivity_90d_percent,inactivity_180d_percent
cutoff_date,,,
2020-03-31,54.21,41.12,20.10
2020-06-30,54.30,41.44,20.03
2020-09-30,54.50,41.46,20.36
2020-12-31,54.79,41.74,20.05
2021-03-31,54.01,41.21,20.09
2021-06-30,54.74,41.90,20.16
2021-09-30,54.06,41.27,20.18
2021-12-31,54.40,41.51,19.82
2022-03-31,54.07,41.49,20.41


## 3. Target Definition

The selected target is transactional inactivity during the 90 days following each cutoff date.

A customer is eligible when at least one transaction was observed during the previous 180 days.

The binary target is defined as:

- `0`: at least one transaction during the following 90 days;
- `1`: no transactions during the following 90 days.

The 90-day window was selected because it provides a balance between business actionability and class distribution.

The resulting inactivity rate is approximately 41.37%.

This target represents future transactional inactivity as a behavioral proxy for churn. It does not represent confirmed account cancellation.

In [7]:
temporal_target = (
    target_window_comparison[
        [
            "customer_id",
            "cutoff_date",
            "transactions_previous_180d",
            "transactions_next_90d",
            "inactivity_90d_label"
        ]
    ]
    .rename(
        columns={
            "inactivity_90d_label":
                "transaction_inactivity_90d_label"
        }
    )
    .copy()
)

temporal_target["cutoff_date"] = pd.to_datetime(
    temporal_target["cutoff_date"]
)

display(temporal_target.head())

,customer_id,cutoff_date,transactions_previous_180d,transactions_next_90d,transaction_inactivity_90d_label
0,CUS000MKX5RHTAP,2020-03-31,1,1,0
1,CUS0015YUPYM2OQ,2020-03-31,3,2,0
2,CUS002V4AVJO5UQ,2020-03-31,5,0,1
3,CUS0031SK48C4X5,2020-03-31,2,0,1
4,CUS003E82EMAYB3,2020-03-31,2,1,0


In [8]:
TARGET = "transaction_inactivity_90d_label"

assert temporal_target.duplicated(
    subset=["customer_id", "cutoff_date"]
).sum() == 0

assert set(
    temporal_target[TARGET].unique()
) == {0, 1}

assert (
    temporal_target["transactions_previous_180d"] > 0
).all()

assert (
    temporal_target.loc[
        temporal_target[TARGET] == 1,
        "transactions_next_90d"
    ] == 0
).all()

assert (
    temporal_target.loc[
        temporal_target[TARGET] == 0,
        "transactions_next_90d"
    ] > 0
).all()

print("Temporal target validation passed!")

display(
    temporal_target[TARGET]
    .value_counts()
    .rename_axis("class")
    .to_frame("customer_snapshots")
)

print(
    "Inactivity rate:",
    f"{temporal_target[TARGET].mean():.2%}"
)

Temporal target validation passed!


,customer_snapshots
class,
0,386125
1,272462


Inactivity rate: 41.37%


## 4. Historical Transaction Features

For each customer and cutoff date, features are calculated exclusively from transactions observed during the previous 180 days.

The feature set captures:

- transaction frequency;
- monetary volume;
- active transaction days;
- transaction recency;
- merchant diversity;
- recent behavioral change.

No transaction after the cutoff date is used to create the features.

In [9]:
temporal_features_query = """
WITH cutoffs AS (

    SELECT
        (
            month_start +
            INTERVAL '1 month' -
            INTERVAL '1 day'
        )::date AS cutoff_date

    FROM GENERATE_SERIES(
        '2020-03-01'::date,
        '2025-06-01'::date,
        INTERVAL '3 months'
    ) AS month_start

),

customer_transactions AS (

    SELECT
        a.customer_id,
        t.merchant_id,
        t.amount_usd,
        t.transaction_date

    FROM dbt.stg_transactions t

    INNER JOIN dbt.stg_accounts a
        ON t.account_id = a.account_id

)

SELECT
    c.cutoff_date,
    ct.customer_id,

    COUNT(*) FILTER (
        WHERE ct.transaction_date >=
            c.cutoff_date - INTERVAL '29 days'
    ) AS transactions_last_30d,

    COUNT(*) FILTER (
        WHERE ct.transaction_date >=
                c.cutoff_date - INTERVAL '59 days'
          AND ct.transaction_date <
                c.cutoff_date - INTERVAL '29 days'
    ) AS transactions_previous_30d,

    COUNT(*) FILTER (
        WHERE ct.transaction_date >=
            c.cutoff_date - INTERVAL '59 days'
    ) AS transactions_last_60d,

    COUNT(*) FILTER (
        WHERE ct.transaction_date >=
            c.cutoff_date - INTERVAL '89 days'
    ) AS transactions_last_90d,

    COUNT(*) AS transactions_last_180d,

    COALESCE(
        SUM(ct.amount_usd) FILTER (
            WHERE ct.transaction_date >=
                c.cutoff_date - INTERVAL '29 days'
        ),
        0
    ) AS amount_last_30d,

    COALESCE(
        SUM(ct.amount_usd) FILTER (
            WHERE ct.transaction_date >=
                    c.cutoff_date - INTERVAL '59 days'
              AND ct.transaction_date <
                    c.cutoff_date - INTERVAL '29 days'
        ),
        0
    ) AS amount_previous_30d,

    COALESCE(
        SUM(ct.amount_usd) FILTER (
            WHERE ct.transaction_date >=
                c.cutoff_date - INTERVAL '89 days'
        ),
        0
    ) AS amount_last_90d,

    SUM(ct.amount_usd) AS amount_last_180d,

    AVG(ct.amount_usd) AS avg_transaction_amount_180d,

    COUNT(
        DISTINCT ct.transaction_date::date
    ) AS active_days_last_180d,

    COUNT(
        DISTINCT ct.merchant_id
    ) AS merchant_diversity_last_180d,

    (
        c.cutoff_date -
        MAX(ct.transaction_date)::date
    ) AS days_since_last_transaction

FROM cutoffs c

INNER JOIN customer_transactions ct
    ON ct.transaction_date >=
        c.cutoff_date - INTERVAL '179 days'
   AND ct.transaction_date <
        c.cutoff_date + INTERVAL '1 day'

GROUP BY
    c.cutoff_date,
    ct.customer_id

ORDER BY
    c.cutoff_date,
    ct.customer_id;
"""

temporal_features = pd.read_sql(
    temporal_features_query,
    engine
)

temporal_features["cutoff_date"] = pd.to_datetime(
    temporal_features["cutoff_date"]
)

print("Shape:", temporal_features.shape)

display(temporal_features.head())

Shape: (658587, 15)


,cutoff_date,customer_id,transactions_last_30d,transactions_previous_30d,transactions_last_60d,transactions_last_90d,transactions_last_180d,amount_last_30d,amount_previous_30d,amount_last_90d,amount_last_180d,avg_transaction_amount_180d,active_days_last_180d,merchant_diversity_last_180d,days_since_last_transaction
0,2020-03-31,CUS000MKX5RHTAP,0,0,0,1,1,0.00,0.00,718.17,718.17,718.170000,1,1,81
1,2020-03-31,CUS0015YUPYM2OQ,0,0,0,0,3,0.00,0.00,0.00,17321.96,5773.986667,2,3,96
2,2020-03-31,CUS002V4AVJO5UQ,3,0,3,4,5,17403.56,0.00,19535.93,20999.65,4199.930000,5,5,9
3,2020-03-31,CUS0031SK48C4X5,0,1,1,1,2,0.00,4743.65,4743.65,11190.51,5595.255000,2,2,42
4,2020-03-31,CUS003E82EMAYB3,0,0,0,1,2,0.00,0.00,586.18,1661.67,830.835000,2,2,76


In [10]:
temporal_features[
    "transaction_change_30d"
] = (
    temporal_features["transactions_last_30d"] -
    temporal_features["transactions_previous_30d"]
)

temporal_features[
    "transaction_change_rate_30d"
] = (
    (
        temporal_features["transactions_last_30d"] + 1
    ) /
    (
        temporal_features["transactions_previous_30d"] + 1
    )
) - 1

temporal_features[
    "amount_change_30d"
] = (
    temporal_features["amount_last_30d"] -
    temporal_features["amount_previous_30d"]
)

temporal_features[
    "recent_transaction_share"
] = (
    temporal_features["transactions_last_30d"] /
    temporal_features["transactions_last_180d"]
)

temporal_features[
    "transactions_per_active_day_180d"
] = (
    temporal_features["transactions_last_180d"] /
    temporal_features["active_days_last_180d"]
)

In [11]:
assert temporal_features.duplicated(
    subset=["customer_id", "cutoff_date"]
).sum() == 0

assert (
    temporal_features["transactions_last_180d"] > 0
).all()

assert (
    temporal_features["transactions_last_30d"]
    <=
    temporal_features["transactions_last_60d"]
).all()

assert (
    temporal_features["transactions_last_60d"]
    <=
    temporal_features["transactions_last_90d"]
).all()

assert (
    temporal_features["transactions_last_90d"]
    <=
    temporal_features["transactions_last_180d"]
).all()

assert (
    temporal_features["active_days_last_180d"]
    <=
    temporal_features["transactions_last_180d"]
).all()

assert temporal_features[
    "days_since_last_transaction"
].between(0, 179).all()

print("Temporal feature validation passed!")

Temporal feature validation passed!


In [12]:
temporal_dataset = temporal_features.merge(
    temporal_target[
        [
            "customer_id",
            "cutoff_date",
            TARGET
        ]
    ],
    on=["customer_id", "cutoff_date"],
    how="inner",
    validate="one_to_one"
)

assert len(temporal_dataset) == len(
    temporal_target
)

assert temporal_dataset[TARGET].mean() == (
    temporal_target[TARGET].mean()
)

print("Temporal dataset shape:", temporal_dataset.shape)
print(
    "Inactivity rate:",
    f"{temporal_dataset[TARGET].mean():.2%}"
)

display(temporal_dataset.head())

Temporal dataset shape: (658587, 21)
Inactivity rate: 41.37%


,cutoff_date,customer_id,transactions_last_30d,transactions_previous_30d,transactions_last_60d,transactions_last_90d,transactions_last_180d,amount_last_30d,amount_previous_30d,amount_last_90d,...,avg_transaction_amount_180d,active_days_last_180d,merchant_diversity_last_180d,days_since_last_transaction,transaction_change_30d,transaction_change_rate_30d,amount_change_30d,recent_transaction_share,transactions_per_active_day_180d,transaction_inactivity_90d_label
0,2020-03-31,CUS000MKX5RHTAP,0,0,0,1,1,0.00,0.00,718.17,...,718.170000,1,1,81,0,0.0,0.00,0.0,1.0,0
1,2020-03-31,CUS0015YUPYM2OQ,0,0,0,0,3,0.00,0.00,0.00,...,5773.986667,2,3,96,0,0.0,0.00,0.0,1.5,0
2,2020-03-31,CUS002V4AVJO5UQ,3,0,3,4,5,17403.56,0.00,19535.93,...,4199.930000,5,5,9,3,3.0,17403.56,0.6,1.0,1
3,2020-03-31,CUS0031SK48C4X5,0,1,1,1,2,0.00,4743.65,4743.65,...,5595.255000,2,2,42,-1,-0.5,-4743.65,0.0,1.0,1
4,2020-03-31,CUS003E82EMAYB3,0,0,0,1,2,0.00,0.00,586.18,...,830.835000,2,2,76,0,0.0,0.00,0.0,1.0,0


In [13]:
IDENTIFIER_COLUMNS = [
    "customer_id",
    "cutoff_date"
]

FEATURE_COLUMNS = [
    column
    for column in temporal_dataset.columns
    if column not in (
        IDENTIFIER_COLUMNS + [TARGET]
    )
]

print("Total de features:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)

Total de features: 18
['transactions_last_30d', 'transactions_previous_30d', 'transactions_last_60d', 'transactions_last_90d', 'transactions_last_180d', 'amount_last_30d', 'amount_previous_30d', 'amount_last_90d', 'amount_last_180d', 'avg_transaction_amount_180d', 'active_days_last_180d', 'merchant_diversity_last_180d', 'days_since_last_transaction', 'transaction_change_30d', 'transaction_change_rate_30d', 'amount_change_30d', 'recent_transaction_share', 'transactions_per_active_day_180d']


In [14]:
feature_validation = pd.DataFrame({
    "dtype": temporal_dataset[
        FEATURE_COLUMNS
    ].dtypes.astype(str),

    "null_values": temporal_dataset[
        FEATURE_COLUMNS
    ].isna().sum(),

    "unique_values": temporal_dataset[
        FEATURE_COLUMNS
    ].nunique()
})

infinite_values = np.isinf(
    temporal_dataset[FEATURE_COLUMNS]
).sum()

constant_features = (
    feature_validation[
        feature_validation["unique_values"] <= 1
    ]
    .index
    .tolist()
)

print(
    "Total de nulos:",
    int(feature_validation["null_values"].sum())
)

print(
    "Total de infinitos:",
    int(infinite_values.sum())
)

print(
    "Features constantes:",
    constant_features
)

display(feature_validation)

Total de nulos: 0
Total de infinitos: 0
Features constantes: []


,dtype,null_values,unique_values
transactions_last_30d,int64,0,7
transactions_previous_30d,int64,0,7
transactions_last_60d,int64,0,10
transactions_last_90d,int64,0,12
transactions_last_180d,int64,0,17
amount_last_30d,float64,0,196596
amount_previous_30d,float64,0,197179
amount_last_90d,float64,0,412307
amount_last_180d,float64,0,518125
avg_transaction_amount_180d,float64,0,515747


In [15]:
target_correlations = (
    temporal_dataset[
        FEATURE_COLUMNS + [TARGET]
    ]
    .corr(method="spearman")[TARGET]
    .drop(TARGET)
    .sort_values()
    .to_frame("spearman_correlation")
)

target_correlations[
    "absolute_correlation"
] = (
    target_correlations[
        "spearman_correlation"
    ].abs()
)

target_correlations = (
    target_correlations
    .sort_values(
        "absolute_correlation",
        ascending=False
    )
)

display(target_correlations)

,spearman_correlation,absolute_correlation
merchant_diversity_last_180d,-0.174963,0.174963
transactions_last_180d,-0.174950,0.174950
active_days_last_180d,-0.174523,0.174523
amount_last_180d,-0.145324,0.145324
transactions_last_90d,-0.116465,0.116465
amount_last_90d,-0.100159,0.100159
transactions_last_60d,-0.092114,0.092114
days_since_last_transaction,0.081077,0.081077
transactions_last_30d,-0.066257,0.066257
transactions_previous_30d,-0.064512,0.064512


In [16]:
temporal_dataset[
    "transaction_frequency_group"
] = pd.cut(
    temporal_dataset[
        "transactions_last_180d"
    ],
    bins=[0, 1, 2, 3, 5, 10, np.inf],
    labels=[
        "1",
        "2",
        "3",
        "4-5",
        "6-10",
        "11+"
    ]
)

inactivity_by_frequency = (
    temporal_dataset
    .groupby(
        "transaction_frequency_group",
        observed=True
    )
    .agg(
        customer_snapshots=(
            TARGET,
            "size"
        ),
        inactivity_rate=(
            TARGET,
            "mean"
        )
    )
)

inactivity_by_frequency[
    "inactivity_rate"
] = (
    inactivity_by_frequency[
        "inactivity_rate"
    ]
    .mul(100)
    .round(2)
)

display(inactivity_by_frequency)

,customer_snapshots,inactivity_rate
transaction_frequency_group,,
1,242673,50.06
2,180966,43.19
3,110999,36.06
4-5,94523,28.65
6-10,28996,19.64
11+,430,7.44


In [17]:
temporal_dataset = temporal_dataset.drop(
    columns=["transaction_frequency_group"]
)

In [18]:
cutoff_dates = pd.DataFrame({
    "cutoff_date": (
        temporal_dataset["cutoff_date"]
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )
})

display(cutoff_dates)

,cutoff_date
0,2020-03-31
1,2020-06-30
2,2020-09-30
3,2020-12-31
4,2021-03-31
5,2021-06-30
6,2021-09-30
7,2021-12-31
8,2022-03-31
9,2022-06-30


## 5. Temporal Train, Validation and Test Split

The dataset is divided chronologically.

Two quarterly cutoff periods are excluded as temporal embargo periods. This prevents future target windows from overlapping with the observation periods used by the following split.

The split is defined as:

- Training: cutoff dates through September 2023;
- Embargo: December 2023;
- Validation: March through September 2024;
- Embargo: December 2024;
- Test: March and June 2025.

Customer identifiers and cutoff dates are preserved for auditing but are not provided as model features.

In [19]:
train_mask = (
    temporal_dataset["cutoff_date"] <=
    pd.Timestamp("2023-09-30")
)

validation_mask = (
    temporal_dataset["cutoff_date"].between(
        pd.Timestamp("2024-03-31"),
        pd.Timestamp("2024-09-30")
    )
)

test_mask = (
    temporal_dataset["cutoff_date"] >=
    pd.Timestamp("2025-03-31")
)

embargo_mask = ~(
    train_mask |
    validation_mask |
    test_mask
)

train_data = temporal_dataset.loc[
    train_mask
].copy()

validation_data = temporal_dataset.loc[
    validation_mask
].copy()

test_data = temporal_dataset.loc[
    test_mask
].copy()

embargo_data = temporal_dataset.loc[
    embargo_mask
].copy()

print(
    "Treino:",
    train_data.shape
)

print(
    "Validação:",
    validation_data.shape
)

print(
    "Teste:",
    test_data.shape
)

print(
    "Embargo:",
    embargo_data.shape
)

Treino: (448919, 21)
Validação: (89934, 21)
Teste: (59902, 21)
Embargo: (59832, 21)


In [20]:
split_summary = pd.DataFrame({
    "split": [
        "train",
        "validation",
        "test",
        "embargo"
    ],

    "rows": [
        len(train_data),
        len(validation_data),
        len(test_data),
        len(embargo_data)
    ],

    "first_cutoff": [
        train_data["cutoff_date"].min(),
        validation_data["cutoff_date"].min(),
        test_data["cutoff_date"].min(),
        embargo_data["cutoff_date"].min()
    ],

    "last_cutoff": [
        train_data["cutoff_date"].max(),
        validation_data["cutoff_date"].max(),
        test_data["cutoff_date"].max(),
        embargo_data["cutoff_date"].max()
    ],

    "inactivity_rate": [
        train_data[TARGET].mean(),
        validation_data[TARGET].mean(),
        test_data[TARGET].mean(),
        embargo_data[TARGET].mean()
    ]
})

split_summary["inactivity_rate"] = (
    split_summary["inactivity_rate"]
    .mul(100)
    .round(2)
)

display(split_summary)

,split,rows,first_cutoff,last_cutoff,inactivity_rate
0,train,448919,2020-03-31,2023-09-30,41.39
1,validation,89934,2024-03-31,2024-09-30,41.43
2,test,59902,2025-03-31,2025-06-30,41.14
3,embargo,59832,2023-12-31,2024-12-31,41.35


In [21]:
X_train = train_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET].astype("int8").copy()

X_validation = (
    validation_data[FEATURE_COLUMNS].copy()
)

y_validation = (
    validation_data[TARGET]
    .astype("int8")
    .copy()
)

X_test = test_data[FEATURE_COLUMNS].copy()
y_test = test_data[TARGET].astype("int8").copy()

assert X_train.index.intersection(
    X_validation.index
).empty

assert X_train.index.intersection(
    X_test.index
).empty

assert X_validation.index.intersection(
    X_test.index
).empty

assert train_data["cutoff_date"].max() < (
    validation_data["cutoff_date"].min()
)

assert validation_data["cutoff_date"].max() < (
    test_data["cutoff_date"].min()
)

print("Temporal split validation passed!")

Temporal split validation passed!


In [22]:
temporal_dataset["dataset_split"] = np.select(
    [
        train_mask,
        validation_mask,
        test_mask
    ],
    [
        "train",
        "validation",
        "test"
    ],
    default="embargo"
)

display(
    temporal_dataset[
        "dataset_split"
    ]
    .value_counts()
    .to_frame("rows")
)

,rows
dataset_split,
train,448919
validation,89934
test,59902
embargo,59832


In [23]:
assert temporal_dataset.duplicated(
    subset=["customer_id", "cutoff_date"]
).sum() == 0

assert temporal_dataset[
    FEATURE_COLUMNS
].isna().sum().sum() == 0

assert np.isinf(
    temporal_dataset[FEATURE_COLUMNS]
).sum().sum() == 0

assert set(
    temporal_dataset[TARGET].unique()
) == {0, 1}

assert set(
    temporal_dataset["dataset_split"].unique()
) == {
    "train",
    "validation",
    "test",
    "embargo"
}

forbidden_future_columns = [
    column
    for column in temporal_dataset.columns
    if "next_" in column.lower()
]

assert forbidden_future_columns == []

assert len(temporal_dataset) == (
    len(train_data) +
    len(validation_data) +
    len(test_data) +
    len(embargo_data)
)

print("Final temporal dataset validation passed!")

Final temporal dataset validation passed!


In [24]:
final_temporal_columns = (
    [
        "customer_id",
        "cutoff_date",
        "dataset_split"
    ]
    +
    FEATURE_COLUMNS
    +
    [TARGET]
)

temporal_dataset_final = (
    temporal_dataset[
        final_temporal_columns
    ]
    .copy()
)

print(
    "Final shape:",
    temporal_dataset_final.shape
)

display(
    temporal_dataset_final.head()
)

Final shape: (658587, 22)


,customer_id,cutoff_date,dataset_split,transactions_last_30d,transactions_previous_30d,transactions_last_60d,transactions_last_90d,transactions_last_180d,amount_last_30d,amount_previous_30d,...,avg_transaction_amount_180d,active_days_last_180d,merchant_diversity_last_180d,days_since_last_transaction,transaction_change_30d,transaction_change_rate_30d,amount_change_30d,recent_transaction_share,transactions_per_active_day_180d,transaction_inactivity_90d_label
0,CUS000MKX5RHTAP,2020-03-31,train,0,0,0,1,1,0.00,0.00,...,718.170000,1,1,81,0,0.0,0.00,0.0,1.0,0
1,CUS0015YUPYM2OQ,2020-03-31,train,0,0,0,0,3,0.00,0.00,...,5773.986667,2,3,96,0,0.0,0.00,0.0,1.5,0
2,CUS002V4AVJO5UQ,2020-03-31,train,3,0,3,4,5,17403.56,0.00,...,4199.930000,5,5,9,3,3.0,17403.56,0.6,1.0,1
3,CUS0031SK48C4X5,2020-03-31,train,0,1,1,1,2,0.00,4743.65,...,5595.255000,2,2,42,-1,-0.5,-4743.65,0.0,1.0,1
4,CUS003E82EMAYB3,2020-03-31,train,0,0,0,1,2,0.00,0.00,...,830.835000,2,2,76,0,0.0,0.00,0.0,1.0,0


In [25]:
from pathlib import Path

processed_path = Path(
    "/home/jovyan/data/processed"
)

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

temporal_output_file = (
    processed_path /
    "customer_transaction_inactivity_temporal_dataset.parquet"
)

temporal_dataset_final.to_parquet(
    temporal_output_file,
    index=False
)

print(
    "Dataset exportado em:",
    temporal_output_file
)

print(
    "Tamanho do arquivo:",
    f"{temporal_output_file.stat().st_size / 1024**2:.2f} MB"
)

Dataset exportado em: /home/jovyan/data/processed/customer_transaction_inactivity_temporal_dataset.parquet
Tamanho do arquivo: 22.69 MB


In [26]:
exported_temporal_dataset = pd.read_parquet(
    temporal_output_file
)

assert exported_temporal_dataset.shape == (
    temporal_dataset_final.shape
)

assert list(
    exported_temporal_dataset.columns
) == list(
    temporal_dataset_final.columns
)

assert exported_temporal_dataset[
    TARGET
].mean() == temporal_dataset_final[
    TARGET
].mean()

print("Exported temporal dataset validation passed!")

Exported temporal dataset validation passed!


In [27]:
split_summary = (
    temporal_dataset_final
    .groupby("dataset_split", observed=True)
    .agg(
        customer_snapshots=("customer_id", "size"),
        unique_customers=("customer_id", "nunique"),
        first_cutoff=("cutoff_date", "min"),
        last_cutoff=("cutoff_date", "max"),
        inactivity_rate=(TARGET, "mean")
    )
    .reset_index()
)

split_summary["inactivity_rate"] = (
    split_summary["inactivity_rate"] * 100
).round(2)

split_summary_output_file = (
    processed_path
    / "customer_transaction_inactivity_split_summary.csv"
)

split_summary.to_csv(
    split_summary_output_file,
    index=False
)

assert split_summary_output_file.exists()

print("Resumo dos splits exportado:")
print(split_summary_output_file)

display(split_summary)

Resumo dos splits exportado:
/home/jovyan/data/processed/customer_transaction_inactivity_split_summary.csv


,dataset_split,customer_snapshots,unique_customers,first_cutoff,last_cutoff,inactivity_rate
0,embargo,59832,35952,2023-12-31,2024-12-31,41.35
1,test,59902,33961,2025-03-31,2025-06-30,41.14
2,train,448919,38826,2020-03-31,2023-09-30,41.39
3,validation,89934,35991,2024-03-31,2024-09-30,41.43


In [28]:
import mlflow
from mlflow import MlflowClient
from mlflow.entities import ViewType

MLFLOW_TRACKING_URI = "http://mlflow:5000"
EXPERIMENT_NAME = "finpulse_temporal_dataset"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

deleted_experiments = client.search_experiments(
    view_type=ViewType.DELETED_ONLY
)

deleted_experiment = next(
    (
        experiment
        for experiment in deleted_experiments
        if experiment.name == EXPERIMENT_NAME
    ),
    None
)

if deleted_experiment is not None:
    client.restore_experiment(deleted_experiment.experiment_id)

    print("Experimento restaurado!")
    print("Experiment ID:", deleted_experiment.experiment_id)
else:
    experiment = mlflow.set_experiment(EXPERIMENT_NAME)

    print("Experimento recriado!")
    print("Experiment ID:", experiment.experiment_id)

Experimento recriado!
Experiment ID: 2


In [29]:
import mlflow.data

MLFLOW_TRACKING_URI = "http://mlflow:5000"
MLFLOW_EXPERIMENT = "finpulse_temporal_dataset"

mlflow.set_tracking_uri(
    MLFLOW_TRACKING_URI
)

mlflow.set_experiment(
    MLFLOW_EXPERIMENT
)

print(
    "MLflow Tracking URI:",
    mlflow.get_tracking_uri()
)

MLflow Tracking URI: http://mlflow:5000


In [30]:
import json

metadata = {
    "dataset_name":
        "customer_transaction_inactivity_temporal_dataset",

    "dataset_version":
        "v1",

    "target":
        TARGET,

    "target_definition":
        "No transactions during the 90 days after the cutoff date.",

    "observation_window_days":
        180,

    "prediction_window_days":
        90,

    "cutoff_frequency":
        "quarterly",

    "first_cutoff":
        str(
            temporal_dataset_final[
                "cutoff_date"
            ].min().date()
        ),

    "last_cutoff":
        str(
            temporal_dataset_final[
                "cutoff_date"
            ].max().date()
        ),

    "total_rows":
        int(len(temporal_dataset_final)),

    "total_customers":
        int(
            temporal_dataset_final[
                "customer_id"
            ].nunique()
        ),

    "feature_count":
        int(len(FEATURE_COLUMNS)),

    "features":
        FEATURE_COLUMNS,

    "identifier_columns": [
        "customer_id",
        "cutoff_date"
    ],

    "split_column":
        "dataset_split",

    "source":
        "dbt.stg_transactions",

    "temporal_limitations": [
        "Customer creation dates are not used for temporal modeling.",
        "Account opening dates are not used for temporal modeling.",
        "Loan start dates are not used for temporal modeling."
    ]
}

metadata_output_file = (
    processed_path /
    "customer_transaction_inactivity_metadata.json"
)

with open(
    metadata_output_file,
    "w",
    encoding="utf-8"
) as metadata_file:
    json.dump(
        metadata,
        metadata_file,
        indent=4,
        ensure_ascii=False
    )

print(
    "Metadados criados em:",
    metadata_output_file
)

assert metadata_output_file.exists()

print("Metadata validation passed!")

Metadados criados em: /home/jovyan/data/processed/customer_transaction_inactivity_metadata.json
Metadata validation passed!


In [31]:
assert temporal_output_file.exists()
assert metadata_output_file.exists()
assert split_summary_output_file.exists()

print("Todos os artefatos estão prontos para o MLflow!")

Todos os artefatos estão prontos para o MLflow!


In [32]:
if mlflow.active_run() is not None:
    mlflow.end_run()

mlflow_dataset = mlflow.data.from_pandas(
    temporal_dataset_final,
    source=str(temporal_output_file),
    targets=TARGET,
    name=(
        "customer_transaction_inactivity_"
        "temporal_dataset_v1"
    )
)

split_counts = (
    temporal_dataset_final[
        "dataset_split"
    ]
    .value_counts()
)

split_inactivity_rates = (
    temporal_dataset_final
    .groupby("dataset_split")[TARGET]
    .mean()
)

with mlflow.start_run(
    run_name="temporal_dataset_v1"
) as run:

    mlflow.set_tags({
        "project": "FinPulse AI",
        "pipeline_stage": "temporal_dataset",
        "dataset_version": "v1",
        "problem_type": "binary_classification",
        "target_type": "transactional_inactivity_proxy",
        "data_source": "dbt.stg_transactions",
        "temporal_validation": "true"
    })

    mlflow.log_params({
        "target": TARGET,
        "observation_window_days": 180,
        "prediction_window_days": 90,
        "cutoff_frequency": "quarterly",
        "first_cutoff": "2020-03-31",
        "last_cutoff": "2025-06-30",
        "feature_count": len(FEATURE_COLUMNS),
        "embargo_1": "2023-12-31",
        "embargo_2": "2024-12-31"
    })

    mlflow.log_metrics({
        "total_customer_snapshots":
            float(len(temporal_dataset_final)),

        "total_customers":
            float(
                temporal_dataset_final[
                    "customer_id"
                ].nunique()
            ),

        "total_cutoff_dates":
            float(
                temporal_dataset_final[
                    "cutoff_date"
                ].nunique()
            ),

        "overall_inactivity_rate":
            float(
                temporal_dataset_final[
                    TARGET
                ].mean()
            ),

        "train_rows":
            float(split_counts.get("train", 0)),

        "validation_rows":
            float(
                split_counts.get(
                    "validation",
                    0
                )
            ),

        "test_rows":
            float(split_counts.get("test", 0)),

        "embargo_rows":
            float(
                split_counts.get(
                    "embargo",
                    0
                )
            ),

        "train_inactivity_rate":
            float(
                split_inactivity_rates.get(
                    "train",
                    0
                )
            ),

        "validation_inactivity_rate":
            float(
                split_inactivity_rates.get(
                    "validation",
                    0
                )
            ),

        "test_inactivity_rate":
            float(
                split_inactivity_rates.get(
                    "test",
                    0
                )
            )
    })

    mlflow.log_input(
        mlflow_dataset,
        context="temporal_dataset"
    )

    mlflow.log_artifact(
        str(temporal_output_file),
        artifact_path="dataset"
    )

    mlflow.log_artifact(
        str(metadata_output_file),
        artifact_path="metadata"
    )

    mlflow.log_artifact(
        str(split_summary_output_file),
        artifact_path="metadata"
    )

    dataset_run_id = run.info.run_id
    dataset_experiment_id = (
        run.info.experiment_id
    )

print("Dataset registrado no MLflow!")
print("Run ID:", dataset_run_id)

print(
    "Abrir no navegador:",
    (
        "http://localhost:5000/"
        f"#/experiments/{dataset_experiment_id}"
        f"/runs/{dataset_run_id}"
    )
)

/opt/conda/lib/python3.11/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

🏃 View run temporal_dataset_v1 at: http://mlflow:5000/#/experiments/2/runs/0d554e2a37644c4b813a21108dd7eb1c
🧪 View experiment at: http://mlflow:5000/#/experiments/2
Dataset registrado no MLflow!
Run ID: 0d554e2a37644c4b813a21108dd7eb1c
Abrir no navegador: http://localhost:5000/#/experiments/2/runs/0d554e2a37644c4b813a21108dd7eb1c
